# S25 — Diffusion Models

**Week 13 · Module 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s25_diffusion_models.ipynb)

Every cell below is a worked example from the [S25 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s25/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s25.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s25.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## The forward process: destroying data on a schedule


*Expected output starts with:* `T = 1000, betas 1e-4 .. 0.02`


In [ ]:
import torch

torch.manual_seed(0)

# Two-moons data, generated in-code and standardized
def two_moons(n):
    t = torch.rand(n) * torch.pi
    outer = torch.stack([torch.cos(t[: n // 2]), torch.sin(t[: n // 2])], dim=1)
    inner = torch.stack([1 - torch.cos(t[n // 2 :]), 0.5 - torch.sin(t[n // 2 :])], dim=1)
    x = torch.cat([outer, inner]) + 0.05 * torch.randn(n, 2)
    return (x - x.mean(0)) / x.std(0)

x0 = two_moons(4000)

def report(T, beta_max, ts):
    betas = torch.linspace(1e-4, beta_max, T)
    alpha_bar = torch.cumprod(1 - betas, dim=0)
    print(f"T = {T}, betas 1e-4 .. {beta_max}")
    print(f"{'t':>4} {'sqrt(a_bar)':>12} {'std(x_t)':>9} {'corr(x_t, x0)':>14}")
    for t in ts:
        eps = torch.randn_like(x0)
        xt = alpha_bar[t].sqrt() * x0 + (1 - alpha_bar[t]).sqrt() * eps
        corr = torch.corrcoef(torch.stack([xt[:, 0], x0[:, 0]]))[0, 1]
        print(f"{t:>4} {alpha_bar[t].sqrt().item():>12.4f} "
              f"{xt.std().item():>9.4f} {corr.item():>14.4f}")

# The DDPM paper's schedule: T=1000, betas from 1e-4 to 0.02
report(1000, 0.02, [0, 249, 499, 749, 999])
print()
# Same beta range but only 200 steps: noising never finishes
report(200, 0.02, [0, 99, 199])
print()
# 200 steps with beta_max rescaled: noising completes again
report(200, 0.05, [0, 99, 199])

## A tiny diffusion model, trained and sampled


*Expected output starts with:* `step    0  denoising loss 0.9931`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def two_moons(n):
    t = torch.rand(n) * torch.pi
    outer = torch.stack([torch.cos(t[: n // 2]), torch.sin(t[: n // 2])], dim=1)
    inner = torch.stack([1 - torch.cos(t[n // 2 :]), 0.5 - torch.sin(t[n // 2 :])], dim=1)
    x = torch.cat([outer, inner]) + 0.05 * torch.randn(n, 2)
    return (x - x.mean(0)) / x.std(0)

data = two_moons(4000)

T = 200
betas = torch.linspace(1e-4, 0.05, T)   # beta_max rescaled so noising completes in 200 steps
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def t_embed(t):  # t: (n,) integer timesteps -> (n, 5) features
    ts = t.float() / T
    return torch.stack([ts, torch.sin(2 * torch.pi * ts), torch.cos(2 * torch.pi * ts),
                        torch.sin(8 * torch.pi * ts), torch.cos(8 * torch.pi * ts)], dim=1)

net = nn.Sequential(nn.Linear(7, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(),
                    nn.Linear(128, 2))          # predicts the noise eps
opt = torch.optim.Adam(net.parameters(), lr=1e-3)

for step in range(4000):
    x0 = data[torch.randint(0, len(data), (256,))]
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(x0)
    xt = alpha_bar[t, None].sqrt() * x0 + (1 - alpha_bar[t, None]).sqrt() * eps
    pred = net(torch.cat([xt, t_embed(t)], dim=1))
    loss = ((pred - eps) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 1000 == 0 or step == 3999:
        print(f"step {step:4d}  denoising loss {loss.item():.4f}")

# Ancestral sampling: start from pure noise, denoise step by step
with torch.no_grad():
    x = torch.randn(2000, 2)
    for t in reversed(range(T)):
        tt = torch.full((2000,), t)
        eps_hat = net(torch.cat([x, t_embed(tt)], dim=1))
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps_hat) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)

def stats(name, s):
    corr = torch.corrcoef(s.T)[0, 1]
    print(f"{name:>10}  mean ({s[:, 0].mean():+.4f}, {s[:, 1].mean():+.4f})  "
          f"std ({s[:, 0].std():.4f}, {s[:, 1].std():.4f})  corr(x1,x2) {corr:+.4f}")

stats("data", data)
stats("samples", x)
stats("N(0, I)", torch.randn(2000, 2))

# Average distance from each sample to its nearest data point
holdout = two_moons(2000)
for name, s in [("held-out data", holdout), ("model samples", x),
                ("N(0, I) noise", torch.randn(2000, 2))]:
    d = torch.cdist(s, data).min(dim=1).values.mean()
    print(f"mean nearest-data distance, {name:>14}: {d.item():.4f}")

## Sampling in fewer steps: DDIM


*Expected output starts with:* `       sampler  NN dist  corr(x1,x2)  (denoiser calls)`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def two_moons(n):
    t = torch.rand(n) * torch.pi
    outer = torch.stack([torch.cos(t[: n // 2]), torch.sin(t[: n // 2])], dim=1)
    inner = torch.stack([1 - torch.cos(t[n // 2 :]), 0.5 - torch.sin(t[n // 2 :])], dim=1)
    x = torch.cat([outer, inner]) + 0.05 * torch.randn(n, 2)
    return (x - x.mean(0)) / x.std(0)

data = two_moons(4000)

T = 200
betas = torch.linspace(1e-4, 0.05, T)
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def t_embed(t):
    ts = t.float() / T
    return torch.stack([ts, torch.sin(2 * torch.pi * ts), torch.cos(2 * torch.pi * ts),
                        torch.sin(8 * torch.pi * ts), torch.cos(8 * torch.pi * ts)], dim=1)

net = nn.Sequential(nn.Linear(7, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(),
                    nn.Linear(128, 2))
opt = torch.optim.Adam(net.parameters(), lr=1e-3)

for step in range(4000):
    x0 = data[torch.randint(0, len(data), (256,))]
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(x0)
    xt = alpha_bar[t, None].sqrt() * x0 + (1 - alpha_bar[t, None]).sqrt() * eps
    pred = net(torch.cat([xt, t_embed(t)], dim=1))
    loss = ((pred - eps) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

def ddpm_sample(n):
    x = torch.randn(n, 2)
    for t in reversed(range(T)):
        tt = torch.full((n,), t)
        eps_hat = net(torch.cat([x, t_embed(tt)], dim=1))
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps_hat) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)
    return x

def ddim_sample(n, n_steps):
    ts = torch.linspace(T - 1, 0, n_steps).long()      # strided timestep subsequence
    x = torch.randn(n, 2)
    for i, t in enumerate(ts):
        tt = torch.full((n,), t.item())
        eps_hat = net(torch.cat([x, t_embed(tt)], dim=1))
        x0_hat = (x - (1 - alpha_bar[t]).sqrt() * eps_hat) / alpha_bar[t].sqrt()
        ab_prev = alpha_bar[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0)
        x = ab_prev.sqrt() * x0_hat + (1 - ab_prev).sqrt() * eps_hat   # eta = 0
    return x

def quality(s):
    d = torch.cdist(s, data).min(dim=1).values.mean().item()
    corr = torch.corrcoef(s.T)[0, 1].item()
    return d, corr

with torch.no_grad():
    holdout = two_moons(2000)
    d, corr = quality(holdout)
    print(f"{'sampler':>14} {'NN dist':>8} {'corr(x1,x2)':>12}  (denoiser calls)")
    print(f"{'held-out data':>14} {d:>8.4f} {corr:>+12.4f}")
    d, corr = quality(ddpm_sample(2000))
    print(f"{'DDPM 200':>14} {d:>8.4f} {corr:>+12.4f}  (200)")
    for k in [50, 20, 10, 5, 2]:
        d, corr = quality(ddim_sample(2000, k))
        print(f"{f'DDIM {k}':>14} {d:>8.4f} {corr:>+12.4f}  ({k})")

## Steering the denoiser: classifier-free guidance


*Expected output starts with:* ` scale  purity  NN dist  spread`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def two_moons_labeled(n):
    t = torch.rand(n) * torch.pi
    outer = torch.stack([torch.cos(t[: n // 2]), torch.sin(t[: n // 2])], dim=1)
    inner = torch.stack([1 - torch.cos(t[n // 2 :]), 0.5 - torch.sin(t[n // 2 :])], dim=1)
    x = torch.cat([outer, inner]) + 0.05 * torch.randn(n, 2)
    y = torch.cat([torch.zeros(n // 2), torch.ones(n // 2)])   # 0 = outer, 1 = inner
    mean, std = x.mean(0), x.std(0)
    return (x - mean) / std, y

data, labels = two_moons_labeled(4000)

T = 200
betas = torch.linspace(1e-4, 0.05, T)
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def t_embed(t):
    ts = t.float() / T
    return torch.stack([ts, torch.sin(2 * torch.pi * ts), torch.cos(2 * torch.pi * ts),
                        torch.sin(8 * torch.pi * ts), torch.cos(8 * torch.pi * ts)], dim=1)

# Conditioning channel: +1 = outer moon, -1 = inner moon, 0 = "no condition" (dropped)
net = nn.Sequential(nn.Linear(8, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(),
                    nn.Linear(128, 2))
opt = torch.optim.Adam(net.parameters(), lr=1e-3)

for step in range(4000):
    idx = torch.randint(0, len(data), (256,))
    x0, y = data[idx], labels[idx]
    c = torch.where(y == 0, 1.0, -1.0)
    c = c * (torch.rand(256) > 0.15).float()      # drop the label 15% of the time
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(x0)
    xt = alpha_bar[t, None].sqrt() * x0 + (1 - alpha_bar[t, None]).sqrt() * eps
    pred = net(torch.cat([xt, t_embed(t), c[:, None]], dim=1))
    loss = ((pred - eps) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

def sample_guided(n, scale, target=1.0):
    """DDPM sampling with classifier-free guidance toward c = target."""
    x = torch.randn(n, 2)
    for t in reversed(range(T)):
        tt = torch.full((n,), t)
        feats = t_embed(tt)
        eps_c = net(torch.cat([x, feats, torch.full((n, 1), target)], dim=1))
        eps_u = net(torch.cat([x, feats, torch.zeros(n, 1)], dim=1))
        eps_hat = eps_u + scale * (eps_c - eps_u)
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps_hat) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)
    return x

outer_pts = data[labels == 0]
with torch.no_grad():
    print(f"{'scale':>6} {'purity':>7} {'NN dist':>8} {'spread':>7}")
    for scale in [0.0, 1.0, 2.0, 4.0, 8.0]:
        s = sample_guided(2000, scale, target=1.0)   # condition: outer moon
        d_all = torch.cdist(s, data)
        nearest = d_all.argmin(dim=1)
        purity = (labels[nearest] == 0).float().mean().item()
        nn_dist = torch.cdist(s, outer_pts).min(dim=1).values.mean().item()
        spread = s.std(dim=0).mean().item()
        print(f"{scale:>6.1f} {purity:>7.3f} {nn_dist:>8.4f} {spread:>7.4f}")
    ref = outer_pts.std(dim=0).mean().item()
    print(f"outer-moon data spread: {ref:.4f}")

## Try it yourself

1. Rerun the sampler with the noise-injection line (`x = x + betas[t].sqrt() * ...`) deleted, and recompute the comparison table. Which statistics degrade, and in which direction?
2. Implement the strided sampler: denoise using only every 4th timestep (50 steps instead of 200), reusing the trained network. How much does the nearest-data distance degrade? This is the intuition behind DDIM-style fast samplers.
3. Replace the linear schedule with a cosine one (`a_bar_t = cos^2((t/T + 0.008)/1.008 * pi/2)`, from Nichol & Dhariwal) and reprint the forward-process table. How does the destruction profile differ at small `t`?
4. Compute the nearest-data distance separately for samples whose nearest data point is on the upper moon vs. the lower moon. Is the model equally good at both — the diffusion analogue of the mode-coverage check from [the GAN section]({{ '/readings/ch4/gans/' | relative_url }})?
5. Add the stochastic term back to DDIM (`eta > 0`): after computing `x0_hat`, mix in fresh noise scaled by `eta * sqrt(1 - ab_prev)` and reduce the deterministic noise share to match. Rebuild the step-count table at `eta` = 0.5 and 1.0. Which step budgets are hurt most by stochasticity?
6. In the guidance experiment, sample with `scale = -1` (guidance *away* from the outer moon) and compute purity with respect to each moon. Relate what you find to negative prompting in image generators.


---

Full discussion of everything above: [S25 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s25/).
